In [1]:
import os
import pandas as pd
import numpy as np

In [2]:
data_path = r"C:\Users\admin\OneDrive\Desktop\healthcare_ppg\data\bidmc_csv"

files = sorted(os.listdir(data_path))
print("Total files:", len(files))
print(files[:10])


Total files: 212
['bidmc_01_Breaths.csv', 'bidmc_01_Fix.txt', 'bidmc_01_Numerics.csv', 'bidmc_01_Signals.csv', 'bidmc_02_Breaths.csv', 'bidmc_02_Fix.txt', 'bidmc_02_Numerics.csv', 'bidmc_02_Signals.csv', 'bidmc_03_Breaths.csv', 'bidmc_03_Fix.txt']


In [3]:
signals = pd.read_csv(os.path.join(data_path, "bidmc_01_Signals.csv"))
numerics = pd.read_csv(os.path.join(data_path, "bidmc_01_Numerics.csv"))
breaths = pd.read_csv(os.path.join(data_path, "bidmc_01_Breaths.csv"))

print("Signals shape:", signals.shape)
print("Numerics shape:", numerics.shape)
print("Breaths shape:", breaths.shape)

print("\nSignals columns:", signals.columns)
print("Numerics columns:", numerics.columns)
print("Breaths columns:", breaths.columns)


Signals shape: (60001, 6)
Numerics shape: (481, 5)
Breaths shape: (170, 2)

Signals columns: Index(['Time [s]', ' RESP', ' PLETH', ' V', ' AVR', ' II'], dtype='object')
Numerics columns: Index(['Time [s]', ' HR', ' PULSE', ' RESP', ' SpO2'], dtype='object')
Breaths columns: Index(['breaths ann1 [signal sample no]', ' breaths ann2 [signal sample no]'], dtype='object')


In [4]:
print("Signal duration (sec):", len(signals)/125)
print("Numerics duration (sec):", len(numerics))


Signal duration (sec): 480.008
Numerics duration (sec): 481


In [6]:
all_rr = []

for i in range(1, 54):
    file = f"bidmc_{i:02d}_Numerics.csv"
    df = pd.read_csv(os.path.join(data_path, file))
    all_rr.extend(df[' RESP'].dropna().values)

all_rr = np.array(all_rr)

print("Min RR:", np.min(all_rr))
print("Max RR:", np.max(all_rr))
print("Mean RR:", np.mean(all_rr))
print("Std RR:", np.std(all_rr))
print("Unique RR values:", len(np.unique(all_rr)))


Min RR: 0.0
Max RR: 34.0
Mean RR: 17.497119179163377
Std RR: 3.444559152423049
Unique RR values: 34


In [9]:
all_rr = []

for i in range(1, 54):
    df = pd.read_csv(os.path.join(data_path, f"bidmc_{i:02d}_Numerics.csv"))
    df.columns = df.columns.str.strip()
    all_rr.extend(df['RESP'].values)

all_rr = np.array(all_rr)

print("Total RR samples:", len(all_rr))
print("Zero RR count:", np.sum(all_rr == 0))
print("Zero RR percentage:", np.mean(all_rr == 0) * 100)

# RR without zeros
rr_clean = all_rr[all_rr > 0]

print("\nAfter removing zeros:")
print("Min RR:", np.min(rr_clean))
print("Max RR:", np.max(rr_clean))
print("Mean RR:", np.mean(rr_clean))
print("Std RR:", np.std(rr_clean))


Total RR samples: 25493
Zero RR count: 220
Zero RR percentage: 0.8629819950574668

After removing zeros:
Min RR: 1.0
Max RR: 34.0
Mean RR: 17.65035828025478
Std RR: 3.0437108645614273


In [14]:
print("RR <= 4 count:", np.sum(rr_clean <= 4))


RR <= 4 count: 11


In [10]:
diff_counts = []

for i in range(1, 54):
    breaths = pd.read_csv(os.path.join(data_path, f"bidmc_{i:02d}_Breaths.csv"))
    breaths.columns = breaths.columns.str.strip()
    
    ann1 = breaths.iloc[:,0].dropna().values
    ann2 = breaths.iloc[:,1].dropna().values
    
    diff_counts.append(abs(len(ann1) - len(ann2)))

print("Max difference in breath counts between annotators:", np.max(diff_counts))
print("Mean difference:", np.mean(diff_counts))


Max difference in breath counts between annotators: 28
Mean difference: 2.2830188679245285


In [11]:
signals = pd.read_csv(os.path.join(data_path, "bidmc_01_Signals.csv"))
signals.columns = signals.columns.str.strip()

breaths = pd.read_csv(os.path.join(data_path, "bidmc_01_Breaths.csv"))
breaths.columns = breaths.columns.str.strip()

numerics = pd.read_csv(os.path.join(data_path, "bidmc_01_Numerics.csv"))
numerics.columns = numerics.columns.str.strip()

# Breath-derived RR (full recording)
breath_count = len(breaths.iloc[:,0].dropna())
rr_breath = breath_count / 8  # 8 minutes recording

# Monitor RR average
rr_monitor = numerics['RESP'][numerics['RESP'] > 0].mean()

print("Breath-derived RR:", rr_breath)
print("Monitor RR (mean):", rr_monitor)


Breath-derived RR: 21.25
Monitor RR (mean): 21.43866943866944


In [12]:
ppg_variances = []

for i in range(1, 54):
    signals = pd.read_csv(os.path.join(data_path, f"bidmc_{i:02d}_Signals.csv"))
    signals.columns = signals.columns.str.strip()
    ppg = signals['PLETH'].values
    ppg_variances.append(np.std(ppg))

print("Min PPG std:", np.min(ppg_variances))
print("Max PPG std:", np.max(ppg_variances))


Min PPG std: 0.0382135271091649
Max PPG std: 0.6734131618808423


In [13]:
fs = 125
window = 30 * fs
stride = 15 * fs

total_samples = 60001

count = 0
start = 0

while start + window <= total_samples:
    count += 1
    start += stride

print("Windows per subject (30s, 50% overlap):", count)
print("Total dataset size:", count * 53)


Windows per subject (30s, 50% overlap): 31
Total dataset size: 1643
